# CARLA adapter check

`carla_raw_data` の 1 frame から `CarlaSensorSample` を作り、`input_adapters.py` の `maptr_input_from_carla()` が MapTR 入力 contract を満たすか確認し、その入力で MapTR 推論まで実行します。

この notebook の主目的は、CARLA raw/schema から MapTR/nuScenes 形式へ変換する interface の入力・出力を明確にすることです。公式 MapTR の正規入力 shape は `01_nuscenes_baseline.ipynb` で確認済みという前提です。


## Data flow

この notebook の入力、途中データ、出力、処理順は [MapTR README](../README.md#02-carla-adapter) にまとめています。

ここでは README の flow に沿って、CARLA raw data から `CarlaSensorSample` を作り、`input_adapters.py` で MapTR 入力へ変換して、pipeline / inference まで確認します。


## CARLA -> MapTR -> planner boundary

MapTR と downstream planner の責務境界は [MapTR README](../README.md#downstream-planner-boundary) にまとめています。

この notebook で直接確認するのは、CARLA 側から MapTR interface に渡す入力と、MapTR へ渡せる形式に変換できているかです。MapTR 出力から planner input への変換は `hdmap-driving-eval` 側で扱います。


## Setup paths: select data from dataset

`dataset/carla_nuscenes/carla_raw_data/runs` を走査して、利用可能な run / frame を一覧表示します。

選び方:
- `RUN_SELECTION_INDEX` で run / repetition を選びます
- `FRAME_SELECTION_INDEX` でその run 内の frame を選びます
- cell 出力の一覧を見て、index を変更して再実行してください

見るところ:
- `Available CARLA raw runs` に使いたい run が出ている
- `Selected RUN_DIR` と `Selected FRAME_ID` が意図したデータになっている
- `metadata`, `ego`, `sensor config` がすべて `exists=True`


In [ ]:
from __future__ import annotations

from pathlib import Path
import gzip
import importlib.util
import json
import math
import sys

WORKSPACE_ROOT = Path('/home/apollo-22/hdmap-driving-workspace')
HDMAP_MODEL_BENCH_ROOT = WORKSPACE_ROOT / 'repos' / 'hdmap-model-bench'
HDMAP_EVAL_ROOT = WORKSPACE_ROOT / 'repos' / 'hdmap-driving-eval'
CARLA_NUSCENES_REPO_ROOT = WORKSPACE_ROOT / 'repos' / 'carla-nuscenes-dataset-builder'
MAPTR_MODEL_DIR = HDMAP_MODEL_BENCH_ROOT / 'models' / 'maptr'
MAPTR_ROOT = MAPTR_MODEL_DIR / 'external' / 'MapTR'
CONFIG_PATH = MAPTR_ROOT / 'projects' / 'configs' / 'maptr' / 'maptr_tiny_r50_24e_bevformer.py'
CHECKPOINT_PATH = WORKSPACE_ROOT / 'checkpoints' / 'maptr' / 'official' / 'maptr_tiny_r50_24e_bevformer.pth'
DEVICE = 'cuda:0'
SCORE_THRESHOLD = 0.30
CARLA_RAW_DATA_ROOT = WORKSPACE_ROOT / 'dataset' / 'carla_nuscenes' / 'carla_raw_data'
CARLA_RAW_RUNS_ROOT = CARLA_RAW_DATA_ROOT / 'runs'
SENSOR_CONFIG_PATH = CARLA_RAW_DATA_ROOT / 'sensor_configs' / 'vehicle_nissan_micra.json'

# Change these two indices after checking the printed list below.
RUN_SELECTION_INDEX = 0
FRAME_SELECTION_INDEX = 0

for path in [HDMAP_MODEL_BENCH_ROOT, HDMAP_EVAL_ROOT, CARLA_NUSCENES_REPO_ROOT, MAPTR_ROOT]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

def discover_raw_runs(runs_root: Path) -> list[dict]:
    candidates = []
    if not runs_root.exists():
        return candidates

    for metadata_dir in sorted(runs_root.glob('*/*/*/*/metadata')):
        run_dir = metadata_dir.parent
        metadata_paths = sorted(
            path for path in metadata_dir.glob('*.json.gz')
            if path.name != 'sensor_status.json'
        )
        frame_ids = [path.stem.split('.')[0] for path in metadata_paths]
        valid_frame_ids = [
            frame_id for frame_id in frame_ids
            if (run_dir / 'ego' / f'{frame_id}.json.gz').exists()
        ]
        if not valid_frame_ids:
            continue
        rel = run_dir.relative_to(runs_root)
        candidates.append({
            'run_dir': run_dir,
            'relative': rel,
            'town': rel.parts[0] if len(rel.parts) > 0 else '',
            'scenario_type': rel.parts[1] if len(rel.parts) > 1 else '',
            'scenario': rel.parts[2] if len(rel.parts) > 2 else '',
            'repetition': rel.parts[3] if len(rel.parts) > 3 else '',
            'frame_ids': valid_frame_ids,
        })
    return candidates

raw_runs = discover_raw_runs(CARLA_RAW_RUNS_ROOT)
if not raw_runs:
    raise FileNotFoundError(f'No usable CARLA raw runs were found under: {CARLA_RAW_RUNS_ROOT}')

print('Available CARLA raw runs:')
for idx, item in enumerate(raw_runs):
    frame_ids = item['frame_ids']
    preview = ', '.join(frame_ids[:8])
    suffix = ' ...' if len(frame_ids) > 8 else ''
    print(
        f'[{idx:02d}] {item["relative"]} '
        f'frames={len(frame_ids)} first=[{preview}{suffix}]'
    )

if not 0 <= RUN_SELECTION_INDEX < len(raw_runs):
    raise IndexError(f'RUN_SELECTION_INDEX={RUN_SELECTION_INDEX} is out of range 0..{len(raw_runs)-1}')
selected_run = raw_runs[RUN_SELECTION_INDEX]
frame_ids = selected_run['frame_ids']
if not 0 <= FRAME_SELECTION_INDEX < len(frame_ids):
    raise IndexError(f'FRAME_SELECTION_INDEX={FRAME_SELECTION_INDEX} is out of range 0..{len(frame_ids)-1}')

RUN_DIR = selected_run['run_dir']
FRAME_ID = frame_ids[FRAME_SELECTION_INDEX]

print()
print('Selected data:')
print('RUN_SELECTION_INDEX:', RUN_SELECTION_INDEX)
print('FRAME_SELECTION_INDEX:', FRAME_SELECTION_INDEX)
print('Selected RUN_DIR:', RUN_DIR)
print('Selected FRAME_ID:', FRAME_ID)

paths = {
    'RUN_DIR': RUN_DIR,
    'metadata': RUN_DIR / 'metadata' / f'{FRAME_ID}.json.gz',
    'ego': RUN_DIR / 'ego' / f'{FRAME_ID}.json.gz',
    'sensor config': SENSOR_CONFIG_PATH,
    'maptr config': CONFIG_PATH,
    'checkpoint': CHECKPOINT_PATH,
}
print()
print('Path checks:')
for name, path in paths.items():
    print(f'{name:14s}: {path} exists={path.exists()}')


## Import dependencies

この notebook は adapter の contract 確認が中心なので、GPU は不要です。画像表示には `cv2` / `matplotlib` を使います。


In [ ]:
required = ['numpy', 'cv2', 'matplotlib']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f'Missing modules in this kernel: {missing}')

import cv2
import numpy as np
import matplotlib.pyplot as plt

from schemas.carla_sensor_sample import (
    CarlaCameraFrame,
    CarlaEgoMotion,
    CarlaIMUFrame,
    CarlaLidarFrame,
    CarlaPose,
    CarlaSensorSample,
)
from models.maptr.interface.input_adapters import DEFAULT_CAMERA_ORDER, maptr_input_from_carla

print('camera order expected by adapter:', DEFAULT_CAMERA_ORDER)


## Load one raw CARLA frame: raw files -> `metadata` / `ego_payload` / `sensor_config`

この cell では `metadata/<frame>.json.gz` と `ego/<frame>.json.gz` を読みます。

見るところ:
- camera が 6 個ある
- `LIDAR_TOP` がある
- `map_name`, `timestamp`, `sample_frame` が意図した frame になっている


In [ ]:
def read_gzip_json(path: Path):
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        return json.load(f)

metadata = read_gzip_json(RUN_DIR / 'metadata' / f'{FRAME_ID}.json.gz')
ego_payload = read_gzip_json(RUN_DIR / 'ego' / f'{FRAME_ID}.json.gz')
sensor_config = json.loads(SENSOR_CONFIG_PATH.read_text(encoding='utf-8'))

frame_metadata = metadata['frame_metadata']
sample_data = metadata['sample_data']
by_channel = {record['channel']: record for record in sample_data}

print('sample_frame:', metadata.get('sample_frame'))
print('timestamp:', frame_metadata.get('timestamp'))
print('timestamp_us:', frame_metadata.get('timestamp_us'))
print('map_name:', frame_metadata.get('map_name'))
print('camera channels:', [record['channel'] for record in sample_data if record['modality'] == 'camera'])
print('lidar channels:', [record['channel'] for record in sample_data if record['modality'] == 'lidar'])
print('ego keys:', ego_payload.keys())


## Build schema input: raw data -> `carla_sample`

sensor config と ego payload から `CarlaSensorSample` を作ります。

raw sensor config には、CARLA attachment (`carla_translation`, `carla_rotation_degrees`) と nuScenes calibrated sensor (`nuscenes_translation`, `nuscenes_rotation`) の両方があります。MapTR の projection は nuScenes camera optical frame を前提にするため、この notebook では default で `nuscenes_*` calibration を使い、それを adapter が受け取る CARLA-style matrix に戻して `sensor_to_world` を作ります。

見るところ:
- `CarlaSensorSample` が 6 camera + 1 lidar を持つ
- camera の順序が `DEFAULT_CAMERA_ORDER`
- image path / lidar path が存在する


In [ ]:
def rotation_matrix_from_quaternion(quat) -> np.ndarray:
    w, x, y, z = [float(v) for v in quat]
    norm = math.sqrt(w*w + x*x + y*y + z*z)
    if norm <= 1e-9:
        w, x, y, z = 1.0, 0.0, 0.0, 0.0
    else:
        w, x, y, z = w / norm, x / norm, y / norm, z / norm
    return np.asarray([
        [1.0 - 2.0 * (y*y + z*z), 2.0 * (x*y - z*w), 2.0 * (x*z + y*w)],
        [2.0 * (x*y + z*w), 1.0 - 2.0 * (x*x + z*z), 2.0 * (y*z - x*w)],
        [2.0 * (x*z - y*w), 2.0 * (y*z + x*w), 1.0 - 2.0 * (x*x + y*y)],
    ], dtype=np.float32)

def matrix_from_translation_rotation(translation, rotation_quat) -> np.ndarray:
    mat = np.eye(4, dtype=np.float32)
    mat[:3, :3] = rotation_matrix_from_quaternion(rotation_quat)
    mat[:3, 3] = np.asarray(translation, dtype=np.float32)
    return mat

def rotation_matrix_from_carla_degrees(rotation: dict) -> np.ndarray:
    roll = math.radians(float(rotation.get('roll', 0.0)))
    pitch = math.radians(float(rotation.get('pitch', 0.0)))
    yaw = math.radians(float(rotation.get('yaw', 0.0)))

    cr, sr = math.cos(roll), math.sin(roll)
    cp, sp = math.cos(pitch), math.sin(pitch)
    cy, sy = math.cos(yaw), math.sin(yaw)

    rx = np.asarray([[1.0, 0.0, 0.0], [0.0, cr, -sr], [0.0, sr, cr]], dtype=np.float32)
    ry = np.asarray([[cp, 0.0, sp], [0.0, 1.0, 0.0], [-sp, 0.0, cp]], dtype=np.float32)
    rz = np.asarray([[cy, -sy, 0.0], [sy, cy, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32)
    return rz @ ry @ rx

def carla_transform_matrix(translation, rotation_degrees) -> np.ndarray:
    mat = np.eye(4, dtype=np.float32)
    mat[:3, :3] = rotation_matrix_from_carla_degrees(rotation_degrees)
    mat[:3, 3] = np.asarray(translation, dtype=np.float32)
    return mat

def carla_matrix_from_nuscenes_matrix(nusc_matrix: np.ndarray) -> np.ndarray:
    flip_y = np.diag(np.asarray([1.0, -1.0, 1.0, 1.0], dtype=np.float32))
    return flip_y @ np.asarray(nusc_matrix, dtype=np.float32) @ flip_y

def sensor_attachment_by_channel(sensor_config: dict) -> dict:
    return {record['channel']: record for record in sensor_config['sensors']}

USE_NUSCENES_CALIBRATION_FROM_RAW_CONFIG = True
attachments = sensor_attachment_by_channel(sensor_config)
ego_to_world = np.asarray(ego_payload['ego_pose']['matrix'], dtype=np.float32)

def sensor_to_ego_matrix_for(channel: str) -> np.ndarray:
    attachment = attachments[channel]
    if USE_NUSCENES_CALIBRATION_FROM_RAW_CONFIG and 'nuscenes_translation' in attachment and 'nuscenes_rotation' in attachment:
        sensor_to_ego_nusc = matrix_from_translation_rotation(
            attachment['nuscenes_translation'],
            attachment['nuscenes_rotation'],
        )
        return carla_matrix_from_nuscenes_matrix(sensor_to_ego_nusc)
    return carla_transform_matrix(
        attachment['carla_translation'],
        attachment['carla_rotation_degrees'],
    )

def sensor_to_world_for(channel: str) -> np.ndarray:
    return ego_to_world @ sensor_to_ego_matrix_for(channel)

cameras = []
for channel in DEFAULT_CAMERA_ORDER:
    record = by_channel[channel]
    attachment = attachments[channel]
    image_path = RUN_DIR / record['filename']
    cameras.append(
        CarlaCameraFrame(
            name=channel,
            image_path=image_path,
            intrinsic=np.asarray(attachment['camera_intrinsic'], dtype=np.float32),
            sensor_to_world=sensor_to_world_for(channel),
            timestamp=record.get('timestamp'),
            width=record.get('width'),
            height=record.get('height'),
            metadata={'raw_record': record},
        )
    )

lidar_record = by_channel['LIDAR_TOP']
lidar = CarlaLidarFrame(
    name='LIDAR_TOP',
    point_path=RUN_DIR / lidar_record['filename'],
    sensor_to_world=sensor_to_world_for('LIDAR_TOP'),
    timestamp=lidar_record.get('timestamp'),
    metadata={'raw_record': lidar_record},
)

can_bus = ego_payload.get('can_bus', {})
imu_values = can_bus.get('imu') or []
imu = CarlaIMUFrame(
    acceleration=imu_values[:3] if len(imu_values) >= 3 else None,
    angular_velocity=imu_values[3:6] if len(imu_values) >= 6 else None,
    timestamp=can_bus.get('timestamp'),
)
ego_motion = CarlaEgoMotion(
    velocity=can_bus.get('velocity'),
    acceleration=can_bus.get('acceleration'),
    angular_velocity=can_bus.get('angular_velocity'),
)

carla_sample = CarlaSensorSample(
    sample_id=f'{RUN_DIR.parent.name}_{RUN_DIR.name}_{FRAME_ID}',
    timestamp=frame_metadata.get('timestamp_us', int(float(frame_metadata.get('timestamp', 0.0)) * 1_000_000)),
    ego_pose=CarlaPose(
        matrix=ego_to_world,
        translation=ego_payload['ego_pose'].get('translation'),
        rotation_quat=ego_payload['ego_pose'].get('rotation'),
        metadata={'rotation_degrees': ego_payload['ego_pose'].get('rotation_degrees')},
    ),
    cameras=tuple(cameras),
    lidar=lidar,
    imu=imu,
    ego_motion=ego_motion,
    map_context={'map_name': frame_metadata.get('map_name')},
    metadata={'run_dir': str(RUN_DIR), 'frame_id': FRAME_ID, 'source': 'carla_raw_data'},
)

print('sample_id:', carla_sample.sample_id)
print('timestamp:', carla_sample.timestamp)
print('calibration mode:', 'nuscenes calibrated sensor from raw config' if USE_NUSCENES_CALIBRATION_FROM_RAW_CONFIG else 'carla attachment')
print('camera names:', [camera.name for camera in carla_sample.cameras])
print('num cameras:', len(carla_sample.cameras))
print('lidar path:', carla_sample.lidar.point_path, Path(carla_sample.lidar.point_path).exists())
for camera in carla_sample.cameras:
    print(f'{camera.name:16s} image_exists={Path(camera.image_path).exists()} intrinsic={np.asarray(camera.intrinsic).shape} sensor_to_world={np.asarray(camera.sensor_to_world).shape}')


## Run adapter: `carla_sample` -> `maptr_input` / `maptr_info`

ここで `maptr_input_from_carla()` を呼び、CARLA raw/schema から MapTR 入力へ変換します。

見るところ:
- `maptr info keys` に MapTR が必要な key が含まれる
- `lidar2img`, `camera_intrinsics`, `camera2ego`, `lidar2ego`, `can_bus` の shape が 01 の baseline と同じ


In [ ]:
maptr_input = maptr_input_from_carla(carla_sample)
maptr_info = maptr_input.to_maptr_info()

print('sample_id:', maptr_input.sample_id)
print('camera names:', [camera.name for camera in maptr_input.cameras])
print('maptr info keys:', sorted(maptr_info.keys()))
print('img_filename length:', len(maptr_info['img_filename']))
print('can_bus:', np.asarray(maptr_info['can_bus']).shape, np.asarray(maptr_info['can_bus']).dtype)
print('lidar2img:', np.asarray(maptr_info['lidar2img']).shape, np.asarray(maptr_info['lidar2img']).dtype)
print('camera_intrinsics:', np.asarray(maptr_info['camera_intrinsics']).shape)
print('camera2ego:', np.asarray(maptr_info['camera2ego']).shape)
print('lidar2ego:', np.asarray(maptr_info['lidar2ego']).shape)


## Contract check: `maptr_info` vs MapTR baseline shape

01 の公式 baseline で確認した MapTR input contract と同じ形かを assert します。

OK の目安:
- すべての check が `True`
- 最後に `CARLA adapter output matches the MapTR input contract.` が出る

ここで落ちた場合は、`input_adapters.py` か raw loader のどちらかに問題があります。


In [ ]:
def check_maptr_contract(info, expected_camera_order=DEFAULT_CAMERA_ORDER):
    required = ['sample_idx', 'pts_filename', 'timestamp', 'can_bus', 'img_filename', 'lidar2img', 'camera_intrinsics', 'camera2ego', 'lidar2ego']
    missing = [key for key in required if key not in info]
    if missing:
        raise AssertionError(f'Missing required keys: {missing}')

    camera_count = len(expected_camera_order)
    checks = {
        'img_filename length': len(info['img_filename']) == camera_count,
        'lidar2img shape': np.asarray(info['lidar2img']).shape == (camera_count, 4, 4),
        'camera_intrinsics shape': np.asarray(info['camera_intrinsics']).shape == (camera_count, 4, 4),
        'camera2ego shape': np.asarray(info['camera2ego']).shape == (camera_count, 4, 4),
        'lidar2ego shape': np.asarray(info['lidar2ego']).shape == (4, 4),
        'can_bus shape': np.asarray(info['can_bus']).shape == (18,),
        'can_bus finite': np.isfinite(np.asarray(info['can_bus'], dtype=float)).all(),
        'lidar2img finite': np.isfinite(np.asarray(info['lidar2img'], dtype=float)).all(),
    }
    for label, passed in checks.items():
        print(f'{label:28s}: {passed}')
        if not passed:
            raise AssertionError(label)

    actual_order = [camera.name for camera in maptr_input.cameras]
    print('actual camera order:', actual_order)
    if tuple(actual_order) != tuple(expected_camera_order):
        raise AssertionError(f'Camera order mismatch: {actual_order}')

check_maptr_contract(maptr_info)
print('CARLA adapter output matches the MapTR input contract.')


## Inspect numeric values: `maptr_info` sanity check

ここでは数値の中身をざっと見ます。

見るところ:
- `can_bus` が finite で、ego pose / velocity / acceleration / yaw らしい値になっている
- `lidar2ego` が極端な値ではない
- `lidar2img` の行列に `nan` や巨大値がない


In [ ]:
print('can_bus:', np.round(np.asarray(maptr_info['can_bus'], dtype=float), 4).tolist())
print('lidar2ego:', np.asarray(maptr_info['lidar2ego']))
print('first camera:', maptr_input.cameras[0].name)
print('first lidar2img:', np.asarray(maptr_info['lidar2img'][0]))


## Visual image order: raw camera images

画像を車両周りの直感的な配置で表示します。

見るところ:
- 上段が `front_left / front / front_right`
- 下段が `back_left / back / back_right`
- 画像の向きが camera 名と合っている

ここで画像が入れ替わって見える場合は、raw metadata の channel 対応や `DEFAULT_CAMERA_ORDER` を疑います。


In [ ]:
image_grid_order = [
    ['CAM_FRONT_LEFT', 'CAM_FRONT', 'CAM_FRONT_RIGHT'],
    ['CAM_BACK_LEFT', 'CAM_BACK', 'CAM_BACK_RIGHT'],
]
image_by_name = {camera.name: Path(camera.image_path) for camera in carla_sample.cameras}

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, camera_name in zip(axes.ravel(), sum(image_grid_order, [])):
    image = cv2.imread(str(image_by_name[camera_name]))
    if image is None:
        raise RuntimeError(f'Cannot read image: {image_by_name[camera_name]}')
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    ax.imshow(image)
    ax.set_title(camera_name)
    ax.axis('off')
plt.tight_layout()


## Projection smoke check: `lidar2ego` / `lidar2img`

ego 座標の簡単な probe points を `lidar2ego` の逆で lidar 座標へ変換し、各 camera 画像へ `lidar2img` で投影します。

見るところ:
- `visible probes` は画像内に入った点数
- `CAM_FRONT` では ego 前方点が見えるのが自然
- 左右 camera で左右反転が疑わしくないかを見る
- `depth` が負なら camera 後方側

これは精密評価ではなく、camera order と `lidar2img` / `lidar2ego` の大きな破綻確認です。


In [ ]:
ego_probe_points = np.asarray([
    [5, 0, 0, 1], [10, 0, 0, 1], [15, 0, 0, 1],
    [10, 5, 0, 1], [10, -5, 0, 1],
], dtype=np.float32)

lidar2ego = np.asarray(maptr_info['lidar2ego'], dtype=np.float32)
ego2lidar = np.linalg.inv(lidar2ego)
lidar_probe_points = (ego2lidar @ ego_probe_points.T).astype(np.float32)

for camera_name, image_path, lidar2img in zip(DEFAULT_CAMERA_ORDER, maptr_info['img_filename'], maptr_info['lidar2img']):
    image = cv2.imread(str(image_path))
    if image is None:
        raise RuntimeError(f'Cannot read image: {image_path}')
    height, width = image.shape[:2]
    projected = np.asarray(lidar2img) @ lidar_probe_points
    depth = projected[2]
    uv = (projected[:2] / np.maximum(depth, 1e-6)).T
    visible = (depth > 0) & (uv[:, 0] >= 0) & (uv[:, 0] < width) & (uv[:, 1] >= 0) & (uv[:, 1] < height)
    print(f'{camera_name:16s} visible probes: {int(visible.sum())}/{len(ego_probe_points)} uv={np.round(uv, 1).tolist()} depth={np.round(depth, 2).tolist()}')


## Build MapTR predictor: config / checkpoint -> `predictor`

ここからは、CARLA raw 由来の `maptr_input` を使って MapTR 推論まで実行します。

見るところ:
- `cuda available: True`
- checkpoint load error が出ない
- `map classes` が `divider`, `ped_crossing`, `boundary` と対応している

ここで落ちる場合は adapter ではなく、MapTR 実行環境・checkpoint・runtime patch 側を疑います。


In [ ]:
import torch
from models.maptr.interface.predictor import MapTRPredictor

print('python:', sys.executable)
print('cuda available:', torch.cuda.is_available())
if not DEVICE.startswith('cuda') or not torch.cuda.is_available():
    raise RuntimeError('Run inference with a GPU-enabled MapTR kernel.')

predictor = MapTRPredictor(
    config_path=CONFIG_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    maptr_root=MAPTR_ROOT,
    device=DEVICE,
    score_threshold=SCORE_THRESHOLD,
)
print('device:', predictor.device)
print('map classes:', predictor.map_classes)


## Inspect pipeline formatting: `maptr_input` -> pipeline data

推論前に、`MapTRSensorInput` が MapTR test pipeline でどのような model input に整形されるか確認します。

見るところ:
- `img` と `img_metas` がある
- `img_metas` に `lidar2img`, `can_bus`, `camera_intrinsics`, `camera2ego`, `lidar2ego` が残っている
- `meta sample_idx` が `maptr_input.sample_id` と一致する


In [ ]:
pipeline_data = predictor._format_sample_for_pipeline(maptr_input)
unwrapped = predictor._unwrap_data_containers(pipeline_data)

def describe(value):
    if hasattr(value, 'shape'):
        return f'{type(value).__name__} shape={tuple(value.shape)} dtype={getattr(value, "dtype", None)}'
    if isinstance(value, list):
        return f'list len={len(value)} first={describe(value[0]) if value else "empty"}'
    if isinstance(value, dict):
        return f'dict keys={list(value.keys())[:16]}'
    return repr(value)[:160]

for key, value in unwrapped.items():
    print(f'{key:12s} -> {describe(value)}')

metas = unwrapped['img_metas']
while isinstance(metas, (list, tuple)):
    metas = metas[0]
print('meta sample_idx:', metas.get('sample_idx'))
print('meta keys:', sorted(metas.keys()))


## Run inference: `maptr_input` -> `raw_output` / `prediction`

この cell が CARLA raw 由来の adapter 出力で実際に MapTR 推論を行う確認です。

見るところ:
- `pts_bbox keys` に `scores_3d`, `labels_3d`, `pts_3d` が含まれる
- `vectors after threshold` が表示される
- `by class` で class ごとの件数が見える


In [ ]:
raw_output = predictor.predict_raw(maptr_input)
prediction = predictor.format_output(
    raw_output,
    sample_id=maptr_input.sample_id,
    score_threshold=SCORE_THRESHOLD,
    raw_output_field=False,
)

detection = predictor._extract_pts_bbox(raw_output)
print('raw type:', type(raw_output))
print('pts_bbox keys:', list(detection.keys()))
for key, value in detection.items():
    print(f'{key:16s} -> {describe(value)}')
print('vectors after threshold:', len(prediction.vectors))
print('by class:', {key: len(value) for key, value in prediction.by_class().items()})


## Visualize BEV prediction

MapTR 出力 vector を ego/lidar BEV 座標で表示します。

見るところ:
- 原点 `ego` が中心付近にある
- x 軸が前方、y 軸が左方向
- 全部が片側に潰れる、左右反転して見える、範囲外に集中する、といった破綻がない


In [ ]:
class_colors = {'divider': '#1f77b4', 'ped_crossing': '#2ca02c', 'boundary': '#d62728'}
pc_range = np.asarray(predictor.cfg.point_cloud_range, dtype=float)
x_min, y_min, _, x_max, y_max, _ = pc_range

fig, ax = plt.subplots(figsize=(7, 10))
ax.set_title(f'MapTR prediction {maptr_input.sample_id} threshold={SCORE_THRESHOLD}')
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_xlabel('x forward [m]')
ax.set_ylabel('y left [m]')
ax.grid(True, alpha=0.25)
ax.set_aspect('equal', adjustable='box')
ax.scatter([0], [0], marker='x', color='black', label='ego')

for vector in prediction.vectors:
    pts = np.asarray(vector.points, dtype=float)
    if len(pts) == 0:
        continue
    ax.plot(
        pts[:, 0],
        pts[:, 1],
        color=class_colors.get(vector.class_name, '#555555'),
        linewidth=1.5,
        alpha=max(0.35, min(1.0, vector.score)),
        label=vector.class_name,
    )
handles, labels = ax.get_legend_handles_labels()
unique = dict(zip(labels, handles))
ax.legend(unique.values(), unique.keys(), loc='upper right')
plt.show()


## Save normalized prediction JSON

CARLA raw 由来の入力に対する `MapTRPrediction.to_dict()` を保存します。

見るところ:
- `outputs/maptr/carla_adapter/<sample_id>.json` の path が表示される
- `sample_token` が `maptr_input.sample_id` と一致する
- `vectors` に `pts`, `pts_num`, `cls_name`, `type`, `confidence_level` が入る


In [ ]:
import json

output_dir = WORKSPACE_ROOT / 'outputs' / 'maptr' / 'carla_adapter'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f'{maptr_input.sample_id}.json'
with output_path.open('w', encoding='utf-8') as f:
    json.dump(prediction.to_dict(), f, indent=2)
print(output_path)
